In [1]:
from model_ranking import (
    mutual_information_from_probs_torch,
    mutual_information_from_probs_np,
    hopkins_statistic,
    uniformity,
    load_h5,
    FeatureExtractor
)
from pytorch3dunet.unet3d.model import UNet2D
from typing import Dict, Any
import torch
import numpy as np
import math

INFO: P [MainThread] 2025-11-04 16:18:34,500 plantseg - Logger configured at initialisation. PlantSeg logger name: plantseg


/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/utils.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# E_model5

In [8]:
model_config: Dict[str, Any] = {
    "name": "UNet2D",
    "in_channels": 1,
    "out_channels": 1,
    "layer_order": "bcr",
    "f_maps": 32,
    "final_sigmoid": True,
    "is_segmentation": True
}
ckpt_path = "/g/kreshuk/talks/segmentation_ModelSelection/experiments/EPFL/BatchNorm/E_model5/best_checkpoint.pytorch"
model = UNet2D(**model_config)
_ = model.load_state_dict(torch.load(ckpt_path)['model_state_dict'])
print(model)

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


UNet2D(
  (encoders): ModuleList(
    (0): Encoder(
      (basic_module): DoubleConv(
        (SingleConv1): SingleConv(
          (batchnorm): BatchNorm2d(1, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (ReLU): ReLU(inplace=True)
        )
        (SingleConv2): SingleConv(
          (batchnorm): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (ReLU): ReLU(inplace=True)
        )
      )
    )
    (1): Encoder(
      (pooling): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (basic_module): DoubleConv(
        (SingleConv1): SingleConv(
          (batchnorm): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv): Conv2d(32, 32, kernel_size=(3, 3), strid

### Uniformity

In [9]:
weight = model.final_conv.weight
#weight = weight.view(weight.size(0), -1)
print("Weight shape:", weight.shape)

Weight shape: torch.Size([1, 32, 1, 1])


In [10]:
weight_2_class = torch.cat([-weight, weight], dim=0)
print("Weight to class shape:", weight_2_class.shape)

Weight to class shape: torch.Size([2, 32, 1, 1])


In [11]:
weight_2_class = weight_2_class.view(weight_2_class.size(0), -1)
print("Weight to class reshaped:", weight_2_class.shape)

Weight to class reshaped: torch.Size([2, 32])


In [15]:
import torch
a = torch.full((1, 32), 0.5)
b = torch.cat([-a, a], dim=0)
unf = uniformity(b)
print("Uniformity:", unf)

Uniformity: 1.1934849908357137e-07


In [12]:
unf = uniformity(weight_2_class)
print(unf)

0.0


# Precomputed Features

In [17]:
EtoE_feature_path = "/g/kreshuk/talks/sampled_features/semantic_segmentation/mitochondria/1k_pixels_sampled/EPFL_to_EPFL/E_model5_to_EPFL_features.h5"
layer_name = "decoders.2"
features_per_patch = load_h5(EtoE_feature_path, f"{layer_name}_features")
labels_per_patch = load_h5(EtoE_feature_path, f"{layer_name}_labels")
predictions_per_patch = load_h5(EtoE_feature_path, f"{layer_name}_predictions")

In [28]:
unique, counts = np.unique(labels_per_patch, return_counts=True)
print("Label distribution:", dict(zip(unique, counts)))

Label distribution: {0.0: 427666, 1.0: 322334}


In [58]:
n_samples_per_class = 10000

# Sample equal number of class 0 and 1 labels
class_0_indices = np.where(labels_per_patch.flatten() == 0)[0]
class_1_indices = np.where(labels_per_patch.flatten() == 1)[0]

# Determine the minimum count between the two classes
min_count = min(len(class_0_indices), len(class_1_indices))

n_samples = max(n_samples_per_class, min_count)

# Randomly sample equal numbers from each class
np.random.seed(42)  # for reproducibility
sampled_class_0 = np.random.choice(class_0_indices, size=n_samples, replace=False)
sampled_class_1 = np.random.choice(class_1_indices, size=n_samples, replace=False)

# Combine the indices
balanced_indices = np.concatenate([sampled_class_0, sampled_class_1])

# Create balanced datasets
features_balanced = features_per_patch.reshape(-1, features_per_patch.shape[-1])[balanced_indices]
labels_balanced = labels_per_patch.flatten()[balanced_indices]
predictions_balanced = predictions_per_patch.flatten()[balanced_indices]

print(f"Original dataset: {len(labels_per_patch.flatten())} samples")
print(f"Balanced dataset: {len(labels_balanced)} samples")
print(f"Class 0: {np.sum(labels_balanced == 0)} samples")
print(f"Class 1: {np.sum(labels_balanced == 1)} samples")

Original dataset: 750000 samples
Balanced dataset: 644668 samples
Class 0: 322334 samples
Class 1: 322334 samples


In [59]:
type(features_balanced)

numpy.ndarray

In [60]:
H_stat = hopkins_statistic(features_balanced)
print("Hopkins statistic:", H_stat)

Hopkins statistic: 0.9876464678891107


In [61]:
predictions_per_patch.shape

(750, 1000)

In [62]:
predictions_per_patch[0, :5]

array([9.9903190e-01, 6.7261295e-05, 9.6159208e-01, 9.9704140e-01,
       6.3517755e-01], dtype=float32)

In [63]:
predictions_balanced.shape

(644668,)

In [64]:
predictions_2_class = np.stack([1 - predictions_balanced, predictions_balanced], axis=1)
print("Predictions to class shape:", predictions_2_class.shape)

Predictions to class shape: (644668, 2)


In [65]:
mi_score = mutual_information_from_probs_np(predictions_2_class)
print("Mutual Information score:", mi_score)

Mutual Information score: -0.6303492188453674


In [68]:
transfer_metric = H_stat - mi_score / math.log(2) - unf
print("Transfer Metric:", transfer_metric)

Transfer Metric: 1.8970481599455542


# V_model2

In [ ]:
VtoE_feature_path = "/g/kreshuk/talks/sampled_features/semantic_segmentation/mitochondria/1k_pixels_sampled/VNC_to_EPFL/V_model2_to_EPFL_features.h5"
layer_name = "decoders.2"
VtoE_features_per_patch = load_h5(VtoE_feature_path, f"{layer_name}_features")
VtoE_labels_per_patch = load_h5(VtoE_feature_path, f"{layer_name}_labels")
VtoE_predictions_per_patch = load_h5(VtoE_feature_path, f"{layer_name}_predictions")

In [ ]:
unique, counts = np.unique(VtoE_labels_per_patch, return_counts=True)
print("Label distribution:", dict(zip(unique, counts)))

Label distribution: {0.0: 427666, 1.0: 322334}


In [ ]:
n_samples_per_class = 10000
import numpy as np
from numpy.typing import NDArray
from typing import Optional, Any


def ensure_even_feature_sampling(
    labels_per_patch: NDArray[Any], 
    features_per_patch: NDArray[Any],
    predictions_per_patch: NDArray[Any],
    n_samples_per_class: Optional[int] = None):
    # Sample equal number of class 0 and 1 labels
    class_0_indices = np.where(labels_per_patch.flatten() == 0)[0]
    class_1_indices = np.where(labels_per_patch.flatten() == 1)[0]

    # Determine the minimum count between the two classes
    min_count = min(len(class_0_indices), len(class_1_indices))

    if n_samples_per_class is None:
        n_samples = min_count
    else:
        n_samples = min_count

    # Randomly sample equal numbers from each class
    np.random.seed(42)  # for reproducibility
    sampled_class_0 = np.random.choice(class_0_indices, size=n_samples, replace=False)
    sampled_class_1 = np.random.choice(class_1_indices, size=n_samples, replace=False)

    # Combine the indices
    balanced_indices = np.concatenate([sampled_class_0, sampled_class_1])

    # Create balanced datasets
    features_balanced = features_per_patch.reshape(-1, features_per_patch.shape[-1])[balanced_indices]
    labels_balanced = labels_per_patch.flatten()[balanced_indices]
    predictions_balanced = predictions_per_patch.flatten()[balanced_indices]

    print(f"Original dataset: {len(labels_per_patch.flatten())} samples")
    print(f"Balanced dataset: {len(labels_balanced)} samples")
    return features_balanced, labels_balanced, predictions_balanced

In [ ]:
VtoE_predictions_per_patch.flatten().ndim

1

In [2]:
def calculate_transfer_metric(
    features: NDArray[Any],
    predictions: NDArray[Any],
    weights: torch.Tensor,
    num_classes: int = 2
):
    H_stat = hopkins_statistic(features)

    if (weights.shape[0] == 1) and (num_classes == 2):
        weights = torch.cat([-weights, weights], dim=0)

    unf = uniformity(weights)

    if (predictions.ndim == 1) and (num_classes == 2):
         predictions = np.stack([1 - predictions, predictions], axis=1)

    mi_score = mutual_information_from_probs_np(predictions)

    transfer_metric = H_stat - mi_score / math.log(2) - unf
    
    return transfer_metric

NameError: name 'NDArray' is not defined

In [3]:
from model_ranking.transferability_metrics.transfer_metrics import (
    get_transfer_data_segmentation,
)
from model_ranking import (
    PrecomputedFeatureConfig,
    PrecomputedDirectPerformanceConfig,
)


feature_cfg = PrecomputedFeatureConfig(
    base_path="/scratch/talks/sampled_features/semantic_segmentation/mitochondria",
    file_type="h5",
    layer_keys = {
        "NA": "decoders.2",
        "Res": "decoders.3",
        "Unetr": "decoder2",
    },
    n_PCA_components=None
)

performance_cfg = PrecomputedDirectPerformanceConfig(
    name="direct_performance",
    base_path="/scratch/talks/consistency_results/patch_segmentation/mitochondria",
    approach="consistency", 
    run_id="P_full",
    key="hard_f1"
)

(
    features,
    predictions,
    labels,
    performance_score,
) = get_transfer_data_segmentation(
    model_name = "V_model2",
    target= "EPFL",
    epoch="",
    feature_config=feature_cfg,
    transferability_metric="Transfer_Score",
    performance_config=performance_cfg,
)

IndentationError: expected an indented block after function definition on line 74 (Transfer_Score.py, line 77)

In [74]:
VtoE_features_balanced, VtoE_labels_balanced, VtoE_predictions_balanced = ensure_even_sampling(
    VtoE_labels_per_patch, VtoE_features_per_patch, VtoE_predictions_per_patch
)

Original dataset: 750000 samples
Balanced dataset: 644668 samples


In [75]:
H_stat = hopkins_statistic(features_balanced)
print("Hopkins statistic:", H_stat)

Hopkins statistic: 0.9878296051053406


In [76]:
predictions_2_class = np.stack([1 - predictions_balanced, predictions_balanced], axis=1)
print("Predictions to class shape:", predictions_2_class.shape)

Predictions to class shape: (644668, 2)


In [77]:
mi_score = mutual_information_from_probs_np(predictions_2_class)
print("Mutual Information score:", mi_score)

Mutual Information score: -0.6303492188453674


In [78]:
transfer_metric = H_stat - mi_score / math.log(2) - unf
print("Transfer Metric:", transfer_metric)

Transfer Metric: 1.8972312971617842
